# SSR — weight-space model merging & spline-support routing

Thin notebook: the ~1100-line *"BUNDLED FROM NOTEBOOK"* prelude that this cell used to carry is gone — it now imports the shared [`nids-toolkit`](https://github.com/braim/nids-toolkit) package. The merging experiment (Parts A–E) below is the original research code, unchanged; a small compatibility adapter threads the `ExperimentConfig` into it.

In [ ]:
# Install the shared toolkit (pulls efficient-kan + the scientific stack).
!pip install -q git+https://github.com/braim/nids-toolkit.git

## Toolkit import + configuration + compatibility adapter

In [ ]:
from nids_toolkit import *                 # config-threaded core API (lazy-loaded)
import nids_toolkit as nt

import gc, copy, itertools
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import f1_score

# ── Experiment configuration (was the bundled 'Section 1' constants) ──────────
cfg = ExperimentConfig()        # defaults reproduce the original SSR settings
device = get_device()

# ── Compatibility shims ───────────────────────────────────────────────────────
# The merging experiment below is the ORIGINAL SSR research code, unchanged. All
# it needs from the old bundled prelude is a few module-level constants and the
# pre-cfg function signatures it was written against — provided here by threading
# `cfg` into the library functions. This replaces ~1100 lines of copy-pasted core.
SEED               = cfg.seed
SAMPLE_N           = cfg.sample_n
LATENT_DIM         = cfg.latent_dim
PRETRAIN_EPOCHS    = cfg.pretrain_epochs
TTA_LR             = cfg.tta_lr
SPLINE_GATE_THRESH = cfg.spline_gate_thresh


def load_dataset(name, sample_n=None, align_to=None):
    return nt.load_dataset(name, cfg, sample_n=sample_n, align_to=align_to)


def make_source_loaders(X, y):
    return nt.make_source_loaders(X, y, cfg)


def make_target_loaders(X, y, external_scaler=None):
    return nt.make_target_loaders(X, y, cfg, external_scaler=external_scaler)


def run_phase1_pretraining(arch, src, input_dim, loader, device, epochs=None, latent_dim=None):
    return nt.run_phase1_pretraining(arch, src, input_dim, loader, device, cfg)


def run_ctta(model, stream_loader, pool_loader, device, **kw):
    return nt.run_ctta(model, stream_loader, pool_loader, device, cfg, **kw)


def make_criterion(y, device):
    return nt.make_criterion(y, device, cfg)


def SplineActivationGate(layer):
    return nt.SplineActivationGate(layer, cfg)

## The merging experiment (Parts A–E)

In [ ]:
# ──────────────────────────────────────────────────────────────────────────
# THE MERGING EXPERIMENT
# ──────────────────────────────────────────────────────────────────────────
import copy, itertools
import pandas as pd

# ── Config ──────────────────────────────────────────────────────────────────
MERGE_ARCH      = 'kan'
MERGE_SOURCE    = 'CICIDS2018'
MERGE_TARGETS   = ['ToN-IoT', 'UNSW-NB15']
MERGE_ALPHAS    = [0.0, 0.25, 0.5, 0.75, 1.0]
MERGE_FAST_EVAL = True        # evaluate merged variants on a 200k-flow
MERGE_EVAL_N    = 200_000     # stratified-ish random subsample of each stream
                              # (adapted/zero-shot endpoints still use full data)

merge_datasets = {
    'CICIDS2018': 'seyhed/nf-cicids2018-v3',
    'ToN-IoT':    'seyhed/nf-ton-iot-v3',
    'UNSW-NB15':  'seyhed/nf-unsw-nb15-v3',
}

def _snap(model):
    """CPU snapshot of a state dict."""
    return {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

def _diff_keys(sd_a, sd_b, tol=0.0):
    """Keys whose tensors differ between two state dicts."""
    return [k for k in sd_a
            if sd_a[k].shape == sd_b[k].shape
            and not torch.equal(sd_a[k], sd_b[k])]

def _interp(sd_src, sd_tgt, alpha):
    """θ(α) = θ_src + α (θ_tgt − θ_src), elementwise over the state dict."""
    out = {k: v.clone() for k, v in sd_src.items()}
    for k in sd_src:
        if sd_src[k].dtype.is_floating_point:
            out[k] = sd_src[k] + alpha * (sd_tgt[k] - sd_src[k])
    return out

def _sub_loader(loader, n=MERGE_EVAL_N):
    """Random n-sample eval loader from a TensorDataset loader (fallback: full)."""
    try:
        xs, ys = loader.dataset.tensors
    except Exception:
        return loader
    if len(ys) <= n or not MERGE_FAST_EVAL:
        return loader
    g = torch.Generator(); g.manual_seed(SEED)
    idx = torch.randperm(len(ys), generator=g)[:n]
    return DataLoader(TensorDataset(xs[idx], ys[idx]),
                      batch_size=1024, shuffle=False)

def _eval_state(model, sd, loader, desc):
    model.load_state_dict(sd)
    return evaluate(model, loader, device, desc=desc)

# ── Phase 0: source pretraining (identical recipe to the grid) ──────────────
seed_everything(SEED)
print('=' * 80)
print(f'EXPERIMENT M | Arch: {MERGE_ARCH.upper()} | Source: {MERGE_SOURCE} | '
      f'Targets: {MERGE_TARGETS}')
print('=' * 80)

X_src, y_src, merge_feats = load_dataset(merge_datasets[MERGE_SOURCE],
                                         sample_n=SAMPLE_N)
input_dim = X_src.shape[1]
loader_src_train, loader_src_test, source_scaler = make_source_loaders(X_src, y_src)
del X_src, y_src; gc.collect()

model = run_phase1_pretraining(MERGE_ARCH, MERGE_SOURCE, input_dim,
                               loader_src_train, device,
                               PRETRAIN_EPOCHS, LATENT_DIM)
sd_src = _snap(model)
f1_src_ceiling = evaluate(model, loader_src_test, device,
                          desc=f'[M] {MERGE_SOURCE} in-domain (source model)')

# ── Phase 1: adapt one copy per target (framework config: spline-only+AGSA),
#             keeping each run's AGSA activation mass for Part C ─────────────
adapted   = {}   # target -> state dict after CTTA
masses    = {}   # target -> AGSA activation-mass tensor (in_dim, n_coeff)
tgt_pool, tgt_stream, tgt_eval = {}, {}, {}

for tgt in MERGE_TARGETS:
    seed_everything(SEED)
    X_t, y_t, _ = load_dataset(merge_datasets[tgt], sample_n=SAMPLE_N,
                               align_to=list(merge_feats))
    pool_l, stream_l = make_target_loaders(X_t, y_t,
                                           external_scaler=source_scaler)
    del X_t, y_t; gc.collect()
    tgt_pool[tgt], tgt_stream[tgt] = pool_l, stream_l
    tgt_eval[tgt] = _sub_loader(stream_l)

    model.load_state_dict(sd_src)
    evaluate(model, stream_l, device, desc=f'[M] zero-shot {tgt}')

    gate = SplineActivationGate(model.encoder.layers[-1])   # external gate:
    run_ctta(model, stream_l, pool_l, device,               # we own its state
             spline_only=True, external_gates=[gate])
    adapted[tgt] = _snap(model)
    masses[tgt]  = gate.mass.detach().cpu().clone()
    gate.remove()

    evaluate(model, stream_l, device, desc=f'[M] adapted {tgt} (full stream)')
    evaluate(model, loader_src_test, device,
             desc=f'[M] {MERGE_SOURCE} after CTTA->{tgt} (forgetting ref)')

src_eval = loader_src_test        # source test set is small; always full

# ── Part A: interpolation curves (zero-data retention dial) ─────────────────
print('\n' + '=' * 80)
print('PART A — source/target trade-off along θ(α), per target')
print('=' * 80)
rows = []
for tgt in MERGE_TARGETS:
    diff = _diff_keys(sd_src, adapted[tgt])
    print(f'[{MERGE_SOURCE}->{tgt}] tensors changed by CTTA: {diff}')
    for a in MERGE_ALPHAS:
        sd = _interp(sd_src, adapted[tgt], a)
        f1_s = _eval_state(model, sd, src_eval,      f'[A] a={a:.2f} src')
        f1_t = _eval_state(model, sd, tgt_eval[tgt], f'[A] a={a:.2f} {tgt}')
        rows.append(dict(part='A', target=tgt, alpha=a, f1_source=f1_s,
                         f1_target=f1_t, f1_min=min(f1_s, f1_t),
                         f1_hmean=2*f1_s*f1_t/max(f1_s+f1_t, 1e-9)))
dfA = pd.DataFrame(rows)
print('\n', dfA.to_string(index=False))
for tgt in MERGE_TARGETS:
    best = dfA[dfA.target == tgt].sort_values('f1_hmean').iloc[-1]
    print(f'[A best joint] {MERGE_SOURCE}<->{tgt}: alpha={best.alpha:.2f} '
          f'src={best.f1_source:.3f} tgt={best.f1_target:.3f}')

# ── Parts B & C: fleet merge of the two adapted copies ──────────────────────
print('\n' + '=' * 80)
print('PART B/C — merge the two adapted models into ONE (no replay, no reset)')
print('=' * 80)
tA, tB   = MERGE_TARGETS
sdA, sdB = adapted[tA], adapted[tB]

# Region masks from the AGSA masses (same rule as the gate itself)
def _active(mass):
    return mass >= SPLINE_GATE_THRESH * mass.max(dim=-1, keepdim=True).values
mA, mB = _active(masses[tA]), _active(masses[tB])           # (in_dim, n_coeff)
inter  = (mA & mB).sum().item(); union = (mA | mB).sum().item()
print(f'[C masks] active {tA}: {mA.float().mean():.1%} | {tB}: '
      f'{mB.float().mean():.1%} | overlap (Jaccard): {inter/max(union,1):.1%}')

def merge_uniform():
    """Model soup: θ_src + (Δ_A + Δ_B)/2."""
    sd = {k: v.clone() for k, v in sd_src.items()}
    for k in set(_diff_keys(sd_src, sdA)) | set(_diff_keys(sd_src, sdB)):
        sd[k] = sd_src[k] + 0.5*(sdA[k]-sd_src[k]) + 0.5*(sdB[k]-sd_src[k])
    return sd

def merge_task_arith():
    """Task arithmetic: θ_src + Δ_A + Δ_B (full-strength sum)."""
    sd = {k: v.clone() for k, v in sd_src.items()}
    for k in set(_diff_keys(sd_src, sdA)) | set(_diff_keys(sd_src, sdB)):
        sd[k] = sd_src[k] + (sdA[k]-sd_src[k]) + (sdB[k]-sd_src[k])
    return sd

def merge_agsa_routed():
    """
    Locality-aware merge (KAN-only): per spline coefficient, the domain that
    ACTIVATED a basis region owns it; contested regions are averaged; regions
    neither domain visited stay at source. Non-spline tensors: uniform mean.
    """
    sd = merge_uniform()   # default for spline_scaler / any other tensor
    for k in _diff_keys(sd_src, sdA):
        if not k.endswith('spline_weight'):
            continue
        if sd_src[k].shape[1:] != mA.shape:       # (out, in, coeff) vs (in, coeff)
            print(f'[C] shape mismatch on {k}, leaving uniform'); continue
        onlyA, onlyB, both = (mA & ~mB), (mB & ~mA), (mA & mB)
        w = sd_src[k].clone()
        w[:, onlyA] = sdA[k][:, onlyA]
        w[:, onlyB] = sdB[k][:, onlyB]
        w[:, both]  = 0.5*sdA[k][:, both] + 0.5*sdB[k][:, both]
        sd[k] = w                                  # dormant regions stay source
    return sd

merge_rows = []
variants = {
    f'adapted->{tA} only': sdA,
    f'adapted->{tB} only': sdB,
    'uniform soup (B)':    merge_uniform(),
    'task arithmetic (B)': merge_task_arith(),
    'AGSA-routed (C)':     merge_agsa_routed(),
}
for name, sd in variants.items():
    f1_s  = _eval_state(model, sd, src_eval,     f'[{name}] {MERGE_SOURCE}')
    f1_a  = _eval_state(model, sd, tgt_eval[tA], f'[{name}] {tA}')
    f1_b  = _eval_state(model, sd, tgt_eval[tB], f'[{name}] {tB}')
    merge_rows.append(dict(part='B/C', variant=name, f1_source=f1_s,
                           **{f'f1_{tA}': f1_a, f'f1_{tB}': f1_b},
                           f1_mean=(f1_s+f1_a+f1_b)/3,
                           f1_min=min(f1_s, f1_a, f1_b)))
dfM = pd.DataFrame(merge_rows)
print('\n', dfM.to_string(index=False))
print('\n[Reference] sequential replay+AGSA endpoint mean (Sec. 9 runs) was '
      '~0.76–0.84 per sequence — but required a sequential pass and stored '
      'pools. The rows above use neither.')

merge_results = dict(interpolation=dfA, fleet=dfM,
                     masks={'A': mA, 'B': mB},
                     source=MERGE_SOURCE, targets=MERGE_TARGETS)
print('\n[M] Done. Results in `merge_results`.')

# ──────────────────────────────────────────────────────────────────────────
# PART D (v2) — SSR: SPLINE-SUPPORT ROUTING with first-layer fingerprints
# Routes each flow to the spline overlay whose RAW-FEATURE grid fingerprint it
# matches (layer 0 = where domains differ; v1's layer-1 hidden space routed at
# only 74-84%). Reports layer-0, layer-1, and combined routers.
# ──────────────────────────────────────────────────────────────────────────
from sklearn.metrics import f1_score as _f1

SSR_EPS = 1e-8
SSR_FP_BATCHES = 200
_first = model.encoder.layers[0]
_last  = model.encoder.layers[-1]
tA, tB = MERGE_TARGETS
names  = ['SRC', tA, tB]

model.load_state_dict(sd_src)

def _fingerprint(layer, loader, n_batches=SSR_FP_BATCHES):
    """Activation-mass profile of `layer` under `loader`'s traffic.
    Layer 0 is frozen+shared, so its fingerprint is expert-independent."""
    # the gate registers a grad hook, which torch refuses on frozen tensors;
    # flip requires_grad on while the gate lives (no grads flow — no_grad pass)
    prev = [(p, p.requires_grad) for p in layer.parameters()]
    for p, _ in prev: p.requires_grad_(True)
    g = SplineActivationGate(layer)
    if hasattr(g, 'capture'): g.capture = True
    model.eval()
    with torch.no_grad():
        for bi, (xb, _) in enumerate(loader):
            model(xb.to(device))
            if bi + 1 >= n_batches: break
    m = g.mass.detach().cpu().clone(); g.remove()
    for p, f in prev: p.requires_grad_(f)
    return m

print('[SSR v2] recording first-layer (raw-feature) fingerprints ...')
mass0 = {'SRC': _fingerprint(_first, loader_src_train),
         tA:    _fingerprint(_first, tgt_stream[tA]),
         tB:    _fingerprint(_first, tgt_stream[tB])}
mass1 = {'SRC': _fingerprint(_last, loader_src_train),
         tA:    masses[tA], tB: masses[tB]}          # layer-1: reuse AGSA masses

def _logp(mass):
    p = mass.clamp(min=0) + SSR_EPS
    return torch.log(p / p.sum(dim=-1, keepdim=True)).to(device)
logp0 = {n: _logp(mass0[n]) for n in names}
logp1 = {n: _logp(mass1[n]) for n in names}

experts = {'SRC': sd_src, tA: adapted[tA], tB: adapted[tB]}
_overlay_keys = [k for k in sd_src if not torch.equal(sd_src[k], adapted[tA][k])]
_params = dict(model.named_parameters())
def _apply_overlay(sd):
    with torch.no_grad():
        for k in _overlay_keys:
            _params[k].data.copy_(sd[k].to(device))

@torch.no_grad()
def ssr_scores(x, mode):
    """Per-sample log-likelihood of each expert's fingerprint. mode: L0|L1|L0+L1."""
    s = 0
    if 'L0' in mode:
        B  = _first.b_splines(x).abs()
        Bn = B / (B.sum(dim=-1, keepdim=True) + SSR_EPS)
        s  = s + torch.stack([(Bn * logp0[n].unsqueeze(0)).sum(dim=(1, 2))
                              for n in names])
    if 'L1' in mode:
        h  = _first(x)
        B  = _last.b_splines(h).abs()
        Bn = B / (B.sum(dim=-1, keepdim=True) + SSR_EPS)
        s  = s + torch.stack([(Bn * logp1[n].unsqueeze(0)).sum(dim=(1, 2))
                              for n in names])
    return s                                          # (n_experts, batch)

@torch.no_grad()
def ssr_evaluate(loader, desc, mode):
    model.eval(); P, Y, R = [], [], []
    for xb, yb in loader:
        xb = xb.to(device)
        route = ssr_scores(xb, mode).argmax(dim=0)
        preds = torch.empty(len(xb), dtype=torch.long)
        for ei, n in enumerate(names):
            sel = (route == ei)
            if not sel.any(): continue
            _apply_overlay(experts[n])
            logits, _, _ = model(xb[sel])
            preds[sel.cpu()] = logits.argmax(1).cpu()
        P.append(preds); Y.append(yb); R.append(route.cpu())
    P, Y, R = torch.cat(P), torch.cat(Y), torch.cat(R)
    f1 = _f1(Y.numpy(), P.numpy(), zero_division=0)
    frac = {n: (R == i).float().mean().item() for i, n in enumerate(names)}
    print(f'[SSR/{mode} {desc}] F1: {f1:.4f} | routed: ' +
          ' '.join(f'{n}={v:.1%}' for n, v in frac.items()))
    return f1, frac

print('\n' + '=' * 78)
print('PART D v2 — SSR routing: layer-0 (raw features) vs layer-1 vs combined')
print('=' * 78)
ssr_rows = []
domain_loaders = [('SRC', loader_src_test), (tA, tgt_eval[tA]), (tB, tgt_eval[tB])]
for mode in ['L0', 'L1', 'L0+L1']:
    row, acc = dict(variant=f'SSR {mode}'), {}
    for dom, loader in domain_loaders:
        f1, frac = ssr_evaluate(loader, dom, mode)
        row[f'f1_{dom}'] = f1; acc[dom] = frac[dom]
    row['f1_mean'] = (row['f1_SRC'] + row[f'f1_{tA}'] + row[f'f1_{tB}']) / 3
    row['router_acc'] = f"{acc['SRC']:.0%}/{acc[tA]:.0%}/{acc[tB]:.0%}"
    ssr_rows.append(row)

dfD = pd.DataFrame(ssr_rows)
print('\n', dfD.to_string(index=False))
print('\n[Reference] best static merge (Part C): mean ~0.50 | '
      'sequential replay+AGSA: mean 0.762. Perfect routing would recover '
      'each expert\'s own-domain F1 (the "adapted (full stream)" lines '
      'above) — SSR stores fingerprints and overlays only, never traffic.')

model.load_state_dict(sd_src)
merge_results['ssr_v2'] = dfD
print('\n[D v2] Done. Results in merge_results["ssr_v2"].')

# ──────────────────────────────────────────────────────────────────────────
# PART D (v3) — WINDOW-SSR: routing at deployment granularity (per window of
# flows, not per flow). Reuses the v2 fingerprints defined above.
# ──────────────────────────────────────────────────────────────────────────
from sklearn.metrics import f1_score as _f1
import matplotlib.pyplot as plt

SSR_EPS = 1e-8
_first = model.encoder.layers[0]
_last  = model.encoder.layers[-1]
tA, tB = MERGE_TARGETS
names  = ['SRC', tA, tB]
model.load_state_dict(sd_src)

# ── fingerprints: reuse from v2 cell if it ran, else compute ────────────────
if 'logp0' not in globals() or 'logp1' not in globals():
    def _fingerprint(layer, loader, n_batches=200):
        prev = [(p, p.requires_grad) for p in layer.parameters()]
        for p, _ in prev: p.requires_grad_(True)
        g = SplineActivationGate(layer)
        if hasattr(g, 'capture'): g.capture = True
        model.eval()
        with torch.no_grad():
            for bi, (xb, _) in enumerate(loader):
                model(xb.to(device))
                if bi + 1 >= n_batches: break
        m = g.mass.detach().cpu().clone(); g.remove()
        for p, f in prev: p.requires_grad_(f)
        return m
    print('[wSSR] recording fingerprints ...')
    mass0 = {'SRC': _fingerprint(_first, loader_src_train),
             tA: _fingerprint(_first, tgt_stream[tA]),
             tB: _fingerprint(_first, tgt_stream[tB])}
    mass1 = {'SRC': _fingerprint(_last, loader_src_train),
             tA: masses[tA], tB: masses[tB]}
    def _logp(m):
        p = m.clamp(min=0) + SSR_EPS
        return torch.log(p / p.sum(dim=-1, keepdim=True)).to(device)
    logp0 = {n: _logp(mass0[n]) for n in names}
    logp1 = {n: _logp(mass1[n]) for n in names}

experts = {'SRC': sd_src, tA: adapted[tA], tB: adapted[tB]}
_overlay_keys = [k for k in sd_src if not torch.equal(sd_src[k], adapted[tA][k])]
_params = dict(model.named_parameters())
def _apply_overlay(sd):
    with torch.no_grad():
        for k in _overlay_keys:
            _params[k].data.copy_(sd[k].to(device))

@torch.no_grad()
def flow_scores(x):
    """Per-flow expert log-likelihoods, layer-0 + layer-1. (n_experts, batch)"""
    B0 = _first.b_splines(x).abs()
    B0 = B0 / (B0.sum(dim=-1, keepdim=True) + SSR_EPS)
    h  = _first(x)
    B1 = _last.b_splines(h).abs()
    B1 = B1 / (B1.sum(dim=-1, keepdim=True) + SSR_EPS)
    return torch.stack([(B0 * logp0[n].unsqueeze(0)).sum(dim=(1, 2)) +
                        (B1 * logp1[n].unsqueeze(0)).sum(dim=(1, 2))
                        for n in names])

# ── (1) router accuracy vs window size ──────────────────────────────────────
print('\n' + '=' * 78)
print('WINDOW-SSR (1) — router accuracy vs window size W')
print('=' * 78)
WINDOWS = [1, 4, 16, 64, 256, 512]
domain_loaders = [('SRC', loader_src_test), (tA, tgt_eval[tA]), (tB, tgt_eval[tB])]
acc_rows = {}
for di, (dom, loader) in enumerate(domain_loaders):
    S = []
    model.eval()
    with torch.no_grad():
        for xb, _ in loader:
            S.append(flow_scores(xb.to(device)).cpu())
    S = torch.cat(S, dim=1)                        # (n_experts, N)
    accs = {}
    for W in WINDOWS:
        n = S.shape[1] // W
        Sw = S[:, :n * W].reshape(S.shape[0], n, W).sum(-1)
        accs[W] = (Sw.argmax(0) == di).float().mean().item()
    acc_rows[dom] = accs
    print(f'  {dom:10s} ' + ' '.join(f'W={W}:{accs[W]:.1%}' for W in WINDOWS))

# ── (2) consolidated F1 with window routing (W = eval batch) ────────────────
print('\n' + '=' * 78)
print('WINDOW-SSR (2) — consolidated model, one routing decision per batch')
print('=' * 78)
rows = []
row = dict(variant='window-SSR (ours)')
for di, (dom, loader) in enumerate(domain_loaders):
    P, Y, hits, nb = [], [], 0, 0
    model.eval()
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            ei = int(flow_scores(xb).sum(dim=1).argmax())   # one decision/window
            hits += (ei == di); nb += 1
            _apply_overlay(experts[names[ei]])
            logits, _, _ = model(xb)
            P.append(logits.argmax(1).cpu()); Y.append(yb)
    f1 = _f1(torch.cat(Y).numpy(), torch.cat(P).numpy(), zero_division=0)
    row[f'f1_{dom}'] = f1
    print(f'[wSSR {dom}] F1: {f1:.4f} | window router accuracy: {hits/nb:.1%} '
          f'({hits}/{nb} windows)')
row['f1_mean'] = (row['f1_SRC'] + row[f'f1_{tA}'] + row[f'f1_{tB}']) / 3
row['f1_min']  = min(row['f1_SRC'], row[f'f1_{tA}'], row[f'f1_{tB}'])
rows.append(row)
dfW = pd.DataFrame(rows)
print('\n', dfW.to_string(index=False))
print('[Reference] sequential replay+AGSA: mean 0.762, min 0.658 | '
      'best static merge: mean ~0.50. Window-SSR stores overlays + '
      'fingerprints only — no traffic, no training, source untouched.')

# ── (3) domain-switching stream: does the router track the change? ──────────
print('\n' + '=' * 78)
print('WINDOW-SSR (3) — switching stream SRC -> ToN -> UNSW (100k flows each)')
print('=' * 78)
SEG_N, WB = 100_000, 512
segs, seg_names = [], ['SRC', tA, tB]
xs_src, ys_src = loader_src_test.dataset.tensors
segs.append((xs_src[:SEG_N], ys_src[:SEG_N]))
for t in [tA, tB]:
    xt, yt = tgt_stream[t].dataset.tensors
    segs.append((xt[:SEG_N], yt[:SEG_N]))
Xs = torch.cat([s[0] for s in segs]); Ys = torch.cat([s[1] for s in segs])
route_trace, f1_trace = [], []
model.eval()
with torch.no_grad():
    for b in range(len(Ys) // WB):
        xb = Xs[b*WB:(b+1)*WB].to(device); yb = Ys[b*WB:(b+1)*WB]
        ei = int(flow_scores(xb).sum(dim=1).argmax())
        route_trace.append(ei)
        _apply_overlay(experts[names[ei]])
        logits, _, _ = model(xb)
        f1_trace.append(_f1(yb.numpy(), logits.argmax(1).cpu().numpy(),
                            zero_division=0))
route_trace = torch.tensor(route_trace)
n_seg = len(route_trace) // 3
for si, sname in enumerate(seg_names):
    seg = route_trace[si*n_seg:(si+1)*n_seg]
    wrong_lead = int((seg != si).float().argmin()) if (seg == si).any() else n_seg
    print(f'  segment {sname:10s}: {(seg == si).float().mean():.1%} of windows '
          f'correctly routed | switch delay: {wrong_lead} window(s)')

fig, ax = plt.subplots(2, 1, figsize=(10, 5), sharex=True,
                       gridspec_kw={'height_ratios': [1, 2]})
ax[0].step(range(len(route_trace)), route_trace, where='post', lw=1.5)
ax[0].set_yticks(range(len(names))); ax[0].set_yticklabels(names)
ax[0].set_ylabel('routed overlay')
for si in (1, 2):
    for a in ax: a.axvline(si * n_seg, color='k', ls='--', lw=1)
ax[1].plot(range(len(f1_trace)), f1_trace, lw=1)
ax[1].set_xlabel(f'window index ({WB} flows/window)')
ax[1].set_ylabel('window F1')
ax[0].set_title('Window-SSR on a domain-switching stream '
                f'({" -> ".join(seg_names)})')
plt.tight_layout(); plt.savefig('fig_ssr_switch.png', dpi=150); plt.show()
print('[wSSR] figure saved: fig_ssr_switch.png')

model.load_state_dict(sd_src)
merge_results['ssr_window'] = dict(acc_vs_window=acc_rows, f1=dfW)
print('\n[D v3] Done. Results in merge_results["ssr_window"].')

# ──────────────────────────────────────────────────────────────────────────
# PART E — JOINT-POOL BASELINE: one model fine-tuned on the union of the
# three domains' labelled pools (300 labels) — the obvious static competitor
# to window-SSR (which uses zero labels at consolidation time).
# ──────────────────────────────────────────────────────────────────────────
from sklearn.model_selection import train_test_split as _tts
from sklearn.metrics import f1_score as _f1j

JOINT_POOL_PER_DOMAIN = 100     # flows per domain (set 50 for the leaner variant)
JOINT_STEPS           = 20      # full-batch steps, matching the few-shot protocol

tA, tB = MERGE_TARGETS

def _pool_tensors(loader, n):
    xs, ys = loader.dataset.tensors
    if len(ys) <= n: return xs, ys
    xp, _, yp, _ = _tts(xs.numpy(), ys.numpy(), train_size=n,
                        random_state=SEED, stratify=ys.numpy())
    return torch.as_tensor(xp), torch.as_tensor(yp)

# source pool: the same 100-flow analyst-labelled assumption as the targets
xs_p, ys_p = _pool_tensors(loader_src_train, JOINT_POOL_PER_DOMAIN)
xa_p, ya_p = _pool_tensors(tgt_pool[tA], JOINT_POOL_PER_DOMAIN)
xb_p, yb_p = _pool_tensors(tgt_pool[tB], JOINT_POOL_PER_DOMAIN)
Xj = torch.cat([xs_p, xa_p, xb_p]).to(device)
Yj = torch.cat([ys_p, ya_p, yb_p]).to(device)
print(f'[Joint] pool: {len(Yj)} flows ({JOINT_POOL_PER_DOMAIN}/domain), '
      f'attack rate {Yj.float().mean():.1%}')

print('\n' + '=' * 78)
print('PART E — joint-pool fine-tune (300 labels) vs window-SSR (0 labels)')
print('=' * 78)
joint_rows = []
for subset_name, param_fn in [('selective', get_trainable_params),
                              ('all params', lambda m: list(m.parameters()))]:
    model.load_state_dict(sd_src)
    params = param_fn(model)
    pids = {id(p) for p in params}
    for p in model.parameters(): p.requires_grad = (id(p) in pids)
    opt  = optim.Adam(params, lr=TTA_LR)
    crit = make_criterion(Yj.cpu(), device)
    model.train()
    for _ in range(JOINT_STEPS):
        opt.zero_grad()
        logits, _, _ = model(Xj)
        loss = crit(logits, Yj)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(params, max_norm=1.0)
        opt.step()
    row = dict(variant=f'joint pool ({subset_name})')
    row['f1_SRC']    = evaluate(model, loader_src_test, device,
                                desc=f'[Joint/{subset_name}] SRC')
    row[f'f1_{tA}']  = evaluate(model, tgt_eval[tA], device,
                                desc=f'[Joint/{subset_name}] {tA}')
    row[f'f1_{tB}']  = evaluate(model, tgt_eval[tB], device,
                                desc=f'[Joint/{subset_name}] {tB}')
    row['f1_mean'] = (row['f1_SRC'] + row[f'f1_{tA}'] + row[f'f1_{tB}']) / 3
    row['f1_min']  = min(row['f1_SRC'], row[f'f1_{tA}'], row[f'f1_{tB}'])
    joint_rows.append(row)

dfE = pd.DataFrame(joint_rows)
print('\n', dfE.to_string(index=False))
print('\n[Reference] window-SSR this configuration: see Part D output '
      '(mean/min above). The joint baseline uses 300 labels at '
      'consolidation time; SSR uses zero.')

model.load_state_dict(sd_src)
merge_results['joint_baseline'] = dfE
print('\n[E] Done. Results in merge_results["joint_baseline"].')